In [ ]:
# 
# encounter level tab data
#labs and aki counts on the encounter level

In [ ]:
# %%
# polars script with datatype toggles
# read_csv
import polars as pl
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
import os
import re
import logging
from tqdm import tqdm
 

# %%
# variables
output_fname = "processed_tab_eskd_v7.csv"
icd_file = "/opt/data/commonfilesharePHI/ldiao/ckd_project/icd_mapping.csv"
subset = False# <<

if subset: 
    subset_size = "1000"  # 10, 100, full # <<
    output_dir = f"/opt/data/workingdir/ldiao/ckd_project/tabular_subset_{subset_size}"
    # output_dir = f"/opt/data/commonfilesharePHI/ldiao/ckd_project/ckd_tab_subset_{subset_size}"
    event_file =  f"/opt/data/workingdir/ldiao/ckd_project/tabular_subset_{subset_size}/unprocessed_tab_subset_{subset_size}.csv"
if not subset:
    # output_dir = "/opt/data/commonfilesharePHI/ldiao/ckd_project/ckd_tab_full"
    output_dir = f"/opt/data/workingdir/ldiao/ckd_project/tabular_full"
    # event_file = "/opt/data/commonfilesharePHI/jnchiang/projects/OptumCKD/CKD-Pull_v2.rpt"
    event_file = "/opt/data/commonfilesharePHI/jnchiang/projects/OptumCKD/CKD-Pull_v3.rpt.parquet"


try:
    os.makedirs(output_dir, exist_ok=True)
    print(f"Created output directory: {output_dir}")
except FileExistsError:
    print(f"Output directory already exists: {output_dir}")

print(f"Processing started. Output directory: {output_dir}")

# %%
# Setup logging
log_file_path = os.path.join(output_dir, "tab_gen_m.log")
logging.basicConfig(
    filename=log_file_path,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

logger.info(f"Processing started. Output directory: {output_dir}")

# %%
# -----------------------------
# Load and preprocess with Polars
# -----------------------------
# Using pl.read_csv to load the entire file into a DataFrame
# Add a toggle to switch between separators
print(event_file)
if subset:
    df = pl.read_csv(
        event_file,
        separator='$',
        infer_schema_length=None,
        null_values=["null", "NULL"],
    ).unique()

    df = df.with_columns(
        pl.col("PatientID").cast(pl.Utf8, strict=False),
        pl.col("EventTimeStamp").cast(pl.Utf8,  strict=False),
        pl.col("DataCategory").cast(pl.Utf8, strict=False),
        pl.col("DataType").cast(pl.Utf8, strict=False),
        pl.col("DataNumeric").cast(pl.Float64, strict=False),
    )
if not subset: 
    csv = event_file 
    df = pl.read_parquet(csv)
    print(len(df['PatientID'].unique()))

logger.info(f"Initial DataFrame schema: {df.schema}")

In [ ]:
df.dtypes

In [ ]:
print(df.shape)

In [ ]:
# -----------------------------
# ICD code -> long title mapping (ported from embedding_gen_pl_v3.py)
# -----------------------------
icd_map_df = pl.read_csv(icd_file)
icd_map_df = icd_map_df.with_columns(
    pl.col("icd_code").cast(pl.Utf8).str.replace(".", "", literal=True)
)
icd_map = dict(zip(icd_map_df["icd_code"], icd_map_df["long_title"]))
logger.info(f"Loaded {len(icd_map)} ICD code -> long_title mappings from {icd_file}")


In [ ]:
icd_map_df.shape

In [ ]:
len(icd_map.keys())

In [ ]:
df.shape

In [ ]:
# Extract and forward-fill ICD

custom_map = {
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    5: 5,
    6: 6,  # ESRD
    9: 0,  # CKD, unspecified stage
}


In [ ]:
print("Building Filter")
# ckd_icd_df's only job now is the patient-level cohort filter (max_stage >= 3).
# Day-level ICD info (icd_daywise, below) is derived from df directly instead,
# since df already has EventDate computed and doesn't need re-plumbing here.
ckd_icd_df = (
    df.filter(pl.col("DataCategory").str.contains("N18"))
    .with_columns(
      pl.col("DataCategory")
        .str.extract(r"N18\.([1-9])", 1)
        .cast(pl.Int64)
        .replace(custom_map, default=None)
        .alias("CKD_stage_numeric")
    )
    .select([pl.col("PatientID"), pl.col("EventTimeStamp"), pl.col("META_1"), pl.col("CKD_stage_numeric")])
    .with_columns(
        pl.col("CKD_stage_numeric")
            .max()
            .over("PatientID")
            .alias("max_stage")
    )
    .filter(pl.col("max_stage") >= 3)
    .unique()
) 


In [ ]:
ckd_icd_df.shape

In [ ]:
# collapse ckd_icd_df from EVENT grain down to (PatientID, META_1) 
ckd_stage_by_encounter = (
    ckd_icd_df
    .group_by(["PatientID", "META_1"])
    .agg([
        pl.col("CKD_stage_numeric").max().alias("CKD_stage_numeric"),  # worst stage diagnosed in this encounter
        pl.col("max_stage").max().alias("max_stage"),  # patient-level max; constant per patient (computed via .over("PatientID") above), .max() here is just a safe collapse
    ])
)

# Sanity check: this must be unique per (PatientID, META_1), or the final join
# will silently fan out again.
n_rows = ckd_stage_by_encounter.shape[0]
n_keys = ckd_stage_by_encounter.select(["PatientID", "META_1"]).unique().shape[0]
assert n_rows == n_keys, f"ckd_stage_by_encounter is not unique per encounter: {n_rows} rows vs {n_keys} unique (PatientID, META_1) keys"
print(f"ckd_stage_by_encounter: {n_rows} rows, one per encounter (confirmed unique)")


In [ ]:
ckd_stage_by_encounter.shape

In [ ]:

print("Filtering and converting")
df = (
    df
    .join(
        ckd_icd_df.select("PatientID").unique(), 
        on="PatientID", 
        how='inner')
    .drop_nulls(subset=["DataNumeric"])
    .with_columns([
        pl.col("EventTimeStamp").str.strptime(pl.Datetime("us")),
        pl.col("DataCategory").fill_null(pl.col("META_2"))
    ])
    .with_columns(
        pl.col("EventTimeStamp").dt.date().alias("EventDate")
    )

)
print(len(df['PatientID'].unique()))
# %% demographics and encounter


In [ ]:
df.shape
# 1394439619,8

In [ ]:
df = df.filter(~pl.col("DataType").is_in(["Demographics", "Encounter"]))

In [ ]:
df.shape

In [ ]:
# Encounter-level skeleton: one row per (PatientID, META_1) encounter.
# icd_daywise is gone -- ckd_icd_df already exists (cell 10) and carries the
# ICD/CKD-stage info we need; we join it in at the end, at encounter grain.
# EventMonth (year-month of the encounter's first date) is the join key used
# below to broadcast monthly-aggregated features (labs, AKI) onto each encounter.



enc_grouped_df = (
    df.group_by(["PatientID", "META_1"])
    .agg(pl.col("EventDate").min().alias("EventDate"))
    .sort(["PatientID", "META_1"])
)
enc_grouped_df.shape



In [ ]:
enc_grouped_df.head

In [ ]:
# %%
aki_icd_codes = ["N17.0", "N17.1", "N17.2", "N17.8", "N17.9"]
aki_events = df.filter(
    (pl.col("DataType") == "Diagnosis") &
    (pl.col("DataCategory").is_in(aki_icd_codes))
).with_columns(
    pl.col("EventDate").dt.strftime("%Y-%m").alias("EventMonth")
)

# Encounter-level AKI count (ICD-code based), keyed on PatientID + META_1 so it
# reflects AKI diagnoses documented within that specific encounter, not
# broadcast across a whole calendar month.
aki_count = aki_events.group_by(["PatientID", "META_1"]).agg([
    pl.len().alias("AKI_ICD_Total"),
])


In [ ]:
print(enc_grouped_df.schema)
print(aki_count.schema)  # and lab_pivot.schema

In [ ]:
# Join encounter-level AKI counts onto the encounter skeleton via PatientID + META_1.
# Each encounter gets the AKI count documented within that same encounter.
enc_grouped_df = enc_grouped_df.join(
    aki_count, on=["PatientID", "META_1"], how="left", join_nulls=True
).with_columns(
    pl.col("AKI_ICD_Total").fill_null(0)  # Ensure encounters with no AKI are 0, not null
)

enc_grouped_df.head()


In [ ]:
# %%
# top lab features from the paper ---
top_lab_features = [
    "CREATININE", "GFR", "GFREST", "ALBUMIN/CREATININE RATIO",
    "PROTEIN/CREATININE RATIO", "BUN", "PTH"
]


lab_df = df.filter(
    (pl.col("DataType") == "Labs") &
    (pl.col("DataCategory").cast(pl.Utf8).str.to_uppercase().str.contains("|".join(top_lab_features)))
).with_columns(
    pl.col("DataCategory").cast(pl.Utf8).str.to_uppercase().alias("LabCategory"),
    pl.col("DataNumeric").cast(pl.Float64, strict=False).alias("DataNumeric"),   # <-- here
)


lab_df = lab_df.sort("EventDate")

# %%

lab_pivot = lab_df.pivot(
    index=["PatientID", "META_1"],
    on="LabCategory",
    values="DataNumeric",
    aggregate_function="first",
)

# %%
lab_pivot


In [ ]:
print(enc_grouped_df.shape)

In [ ]:
# %%
# Prefix all pivoted lab-stat columns with "lab_" for clarity downstream.
rename_dict = {c: f"lab_{c}" for c in lab_pivot.columns if c not in ("PatientID", "META_1")}
lab_pivot = lab_pivot.rename(rename_dict)

enc_grouped_df = enc_grouped_df.join(lab_pivot, on=["PatientID", "META_1"], how="left", join_nulls=True)


In [ ]:

agg_cols = [c for c in enc_grouped_df.columns if c.startswith("lab_") or c == "AKI_ICD_Total"]
enc_grouped_df = enc_grouped_df.with_columns([
    pl.col(c).cast(pl.Float64) for c in agg_cols
])
print(f"Coerced {len(agg_cols)} aggregate columns to Float64: {agg_cols[:5]}{'...' if len(agg_cols) > 5 else ''}")


In [ ]:
# Merge encounter level CKD/ICD-stage

pre_join_rows = enc_grouped_df.shape[0]

enc_grouped_df = enc_grouped_df.join(
    ckd_stage_by_encounter,
    on=["PatientID", "META_1"], how='left'
)

assert enc_grouped_df.shape[0] == pre_join_rows, (
    f"CKD join changed row count: {pre_join_rows} -> {enc_grouped_df.shape[0]}. "
    "ckd_stage_by_encounter is no longer unique per (PatientID, META_1) -- check upstream."
)
print(f"enc_grouped_df: {enc_grouped_df.shape[0]} rows after CKD join (unchanged from {pre_join_rows})")


In [ ]:
print(enc_grouped_df.shape)
base_df = enc_grouped_df

In [ ]:
# -----------------------------
# Final report
# -----------------------------
base_df = enc_grouped_df  # keep the rest of this cell unchanged below

logger.info(f"[INFO] Final tabular shape: {base_df.shape}")
logger.info(f"[INFO] Sample features:\n{base_df.head()}")
logger.info(f"[INFO] CKD stage counts:\n{base_df['CKD_stage_numeric'].value_counts(sort=True)}")
base_df_path = os.path.join(output_dir, output_fname)
logger.info(f"Writing final DataFrame of shape {base_df.shape} to {base_df_path}")
base_df.write_csv(base_df_path)

logger.info("End of Tabular Generation")

# check csv
# Construct the full file path
final_file_path = os.path.join(output_dir, output_fname)
print(final_file_path)
# Read the processed CSV file
try:
    final_df = pl.read_csv(final_file_path)
    print("File read successfully.")
    print(final_df.head())
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

In [ ]:
# %%
import polars as pl
tab_path = f"./tabular_full/{output_fname}"    
df = pl.read_csv(tab_path)

# Overall row/patient counts
print(f"Rows: {df.shape[0]}, Patients: {df['PatientID'].n_unique()}")

# Prevalence of CKD_stage (row-level and patient-level, since patients can span stages)
print(df["CKD_stage_numeric"].value_counts(sort=True).with_columns(
    (pl.col("count") / df.shape[0] * 100).round(2).alias("pct_rows")
))

# Patient-level prevalence: each patient's max (worst) CKD_stage
patient_stage = df.group_by("PatientID").agg(pl.col("CKD_stage_numeric").max())
print(patient_stage["CKD_stage_numeric"].value_counts(sort=True).with_columns(
    (pl.col("count") / patient_stage.shape[0] * 100).round(2).alias("pct_patients")
))



In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df

In [ ]:
len(df['PatientID'].unique())

In [ ]:
prediction_period = 365 # 365, 730, 1095
years = str(round(prediction_period/365))
exclude_cols = ['PatientID', 
                    'EventDate', 
                    'EventMonth',
                    'CKD_stage', # Raw stage column
                    'CKD_stage_clean', # Intermediate cleaned stage
                    'CKD_stage_numeric',
                    'max_stage',
                    "META_1",
                    'label_ckd_stage_4_plus', f'label_ckd_{years}_year_future', # Generated labels
                    'time_until_progression', 'event_for_cox_indicator'] # Generated TTE info
    

In [ ]:
df.columns

In [ ]:
potential = list(set(df.columns) - set(exclude_cols))
potential

In [ ]:
import numpy as np
[col for col in potential if df[col].dtype in [np.number, 'bool']]

In [ ]:
df.head()